In [3]:
# importing dataset obtained from osm 

import pandas as pd
# "C:\Users\ASUS\Downloads\osm_baseline.csv"
# Read the CSV into a DataFrame
df1 = pd.read_csv(r"C:\Users\ASUS\Downloads\osm_baseline.csv")

# Display the first 5 rows
df1.tail()

,Unnamed: 0,Place ID (OSM ID),lat,lon,Place Name,Category Tag,Place Type,Address Information,Contact Details,Operational Info,Metadata
8605,8605,18007977,NaN,NaN,Helix Bridge,tourism,attraction,NaN,NaN,NaN,NaN
8606,8606,18841366,NaN,NaN,PARKTOWN,shop,mall,NaN,NaN,NaN,NaN
8607,8607,19216100,NaN,NaN,The Coffee Bean & Tea Leaf,amenity,cafe,NaN,NaN,NaN,NaN
8608,8608,19216101,NaN,NaN,Fish & Co.,amenity,restaurant,NaN,NaN,NaN,NaN
8609,8609,19914562,NaN,NaN,Season Live Seafood,amenity,restaurant,NaN,NaN,NaN,NaN


In [4]:
# importing dataset obtained from geoapify api
import pandas as pd

# Read the CSV into a DataFrame
df2 = pd.read_csv(r"C:\Users\ASUS\singapore_poi_cleaned.csv")

# Display the first 5 rows
df2.head()

,place_id,name,category,lat,lon,address,city,country
0,517b45fee9a2f6594059c0764cca7ce1f43ff00102f901...,Bismillah Biryani,catering.restaurant,1.305051,103.853693,"Bismillah Biryani, 50 Dunlop Street, Singapore...",Singapore,Singapore
1,51e3a915a35df6594059943af9e05435f53ff00102f901...,Boon Tong Kee,catering.restaurant,1.325520,103.849465,"Boon Tong Kee, 399,401,403 Balestier Road, Sin...",Singapore,Singapore
2,5181481a3790fb5940594893a5c3e2e1f43ff00103f901...,Jumbo Seafood,catering.restaurant,1.305148,103.930677,"Jumbo Seafood, 1206 East Coast Parkway, Singap...",Singapore,Singapore
3,51f1c528194af6594059a5ad3741cea0f43ff00103f901...,Jumbo Seafood,catering.restaurant,1.289259,103.848273,"Jumbo Seafood, 20 Upper Circular Road, Singapo...",Singapore,Singapore
4,51bc56e7bdd6f3594059c3a25fed6cdef43ff00103f901...,Jumbo Seafood @ Dempsey,catering.restaurant,1.304303,103.809982,"Jumbo Seafood @ Dempsey, 11 Dempsey Road, Sing...",Singapore,Singapore


In [11]:
# comparing osm and geoapify datasets

import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# -----------------------------
# 1. Haversine Distance Function
# -----------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c  # distance in km


# -----------------------------
# 2. Clean Name Function
# -----------------------------
def clean_name(name):
    return str(name).lower().strip()


df1['clean_name'] = df1['Place Name'].apply(clean_name)
df2['clean_name'] = df2['name'].apply(clean_name)


# -----------------------------
# 3. Extract lat/lon from df1.location
# -----------------------------
# Assuming location = {'lat': xx, 'lng': xx}
# df1['lat'] = df1['location'].apply(lambda x: x.get('lat') if isinstance(x, dict) else None)
# df1['lon'] = df1['location'].apply(lambda x: x.get('lng') if isinstance(x, dict) else None)


# -----------------------------
# 4. Matching Logic
# -----------------------------
THRESHOLD_KM = 0.1  # 100 meters


def check_match(row):
    name = row['clean_name']
    lat1 = row['lat']
    lon1 = row['lon']

    # -------------------------
    # Step 1: Exact Name Match
    # -------------------------
    name_match = df2[df2['clean_name'] == name]

    if not name_match.empty:
        return "Exists"

    # -------------------------
    # Step 2: Geo Match
    # -------------------------
    if pd.isna(lat1) or pd.isna(lon1):
        return "Not Found"

    for _, cand in df2.iterrows():
        lat2 = cand['lat']
        lon2 = cand['lon']

        if pd.isna(lat2) or pd.isna(lon2):
            continue

        dist = haversine(lat1, lon1, lat2, lon2)

        if dist <= THRESHOLD_KM:
            return "Exists (Geo Match)"

    return "Not Found"


# -----------------------------
# 5. Apply Matching
# -----------------------------
df1['status'] = df1.apply(check_match, axis=1)


# -----------------------------
# 6. Final Output
# -----------------------------
result_df = df1.drop(columns=['clean_name'])

result_df.head()


,Unnamed: 0,Place ID (OSM ID),lat,lon,Place Name,Category Tag,Place Type,Address Information,Contact Details,Operational Info,Metadata,status
0,0,319830199,1.338296,103.730060,Bonsai Garden,tourism,attraction,NaN,NaN,NaN,NaN,Exists
1,1,368261001,1.389365,103.886287,NaN,amenity,cafe,NaN,NaN,NaN,NaN,Not Found
2,2,369123892,1.385612,103.902577,NTUC Fairprice,shop,supermarket,NaN,NaN,24/7,NaN,Exists
3,3,369127276,1.379434,103.887950,Giant,shop,supermarket,"21, Hougang Street 51, Singapore, 538719",NaN,NaN,NaN,Exists
4,4,369128055,1.372528,103.893924,NTUC FairPrice,shop,supermarket,NaN,NaN,08:00-23:00,NaN,Exists


In [12]:
print(result_df['status'].unique())

['Exists' 'Not Found' 'Exists (Geo Match)']


In [17]:
compare_df = result_df

In [18]:
compare_df.tail()

,Unnamed: 0,Place ID (OSM ID),lat,lon,Place Name,Category Tag,Place Type,Address Information,Contact Details,Operational Info,Metadata,status
8605,8605,18007977,NaN,NaN,Helix Bridge,tourism,attraction,NaN,NaN,NaN,NaN,Exists
8606,8606,18841366,NaN,NaN,PARKTOWN,shop,mall,NaN,NaN,NaN,NaN,Exists
8607,8607,19216100,NaN,NaN,The Coffee Bean & Tea Leaf,amenity,cafe,NaN,NaN,NaN,NaN,Exists
8608,8608,19216101,NaN,NaN,Fish & Co.,amenity,restaurant,NaN,NaN,NaN,NaN,Exists
8609,8609,19914562,NaN,NaN,Season Live Seafood,amenity,restaurant,NaN,NaN,NaN,NaN,Exists


In [5]:
# importing datasets obtained from tomtom api

import pandas as pd
# tomtom--------api
# Replace with your file path
# file_path = 

# Read the CSV into a DataFrame
df3 = pd.read_csv(r"C:\Users\ASUS\tomtom_singapore_poi.csv")

# Display the first 5 rows
df3.head()

,place_id,name,category,lat,lon,address
0,lx0bPzJIjCr1DZefcTMMAw,Phbjv-Dormitory,['restaurant'],1.235298,103.616796,"Singapore, 63"
1,spRgvz4K_ub_1f18GaxAfg,Greenhouse Bistro,"['american', 'restaurant']",1.315293,103.631750,"30 Tuas Bay Drive, Singapore, 63"
2,jtCxdxqXQRIq0qn0R06lPQ,Hi-Five Restaurant & Catering Point East,"['fusion', 'restaurant']",1.310741,103.630171,"14 Tech Park Crescent, Singapore, 63"
3,kCT7I63Xdt9-a091PaVB3g,Wow West Food Loft,['restaurant'],1.308100,103.631550,"2 Tuas South Street 2, Singapore, 637895"
4,SNPGiWts5H7-PqBcrAmAcw,Allied Vending Hot Food Machine,['restaurant'],1.306700,103.664900,"2 Pioneer Sector 3, Singapore, 62"


In [19]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# -----------------------------
# 1. Haversine Distance Function
# -----------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c


# -----------------------------
# 2. Clean Name Function
# -----------------------------
def clean_name(name):
    return str(name).lower().strip()


df1['clean_name'] = df1['Place Name'].apply(clean_name)
df3['clean_name'] = df3['name'].apply(clean_name)


# -----------------------------
# 3. Extract lat/lon from df1.location
# -----------------------------
# df1['lat'] = df1['location'].apply(
#     lambda x: x.get('lat') if isinstance(x, dict) else None
# )
# df1['lon'] = df1['location'].apply(
#     lambda x: x.get('lng') if isinstance(x, dict) else None
# )


# -----------------------------
# 4. Matching Logic
# -----------------------------
THRESHOLD_KM = 0.1  # 100 meters


def check_match(row):
    name = row['clean_name']
    lat1 = row['lat']
    lon1 = row['lon']
    place_id = row['Place ID (OSM ID)']

    # -------------------------
    # Step 1: Place ID Match
    # -------------------------
    id_match = df3[df3['place_id'] == place_id]
    if not id_match.empty:
        return "Exists (ID Match)"

    # -------------------------
    # Step 2: Name Match
    # -------------------------
    name_match = df3[df3['clean_name'] == name]
    if not name_match.empty:
        return "Exists"

    # -------------------------
    # Step 3: Geo Match
    # -------------------------
    if pd.isna(lat1) or pd.isna(lon1):
        return "Not Found"

    for _, cand in df3.iterrows():
        lat2 = cand['lat']
        lon2 = cand['lon']

        if pd.isna(lat2) or pd.isna(lon2):
            continue

        dist = haversine(lat1, lon1, lat2, lon2)

        if dist <= THRESHOLD_KM:
            return "Exists (Geo Match)"

    return "Not Found"


# -----------------------------
# 5. Apply Matching
# -----------------------------
df1['status'] = df1.apply(check_match, axis=1)


# -----------------------------
# 6. Final Output
# -----------------------------
result_df = df1.drop(columns=['clean_name'])

print(result_df.head())

   Unnamed: 0  Place ID (OSM ID)       lat         lon      Place Name  \
0           0          319830199  1.338296  103.730060   Bonsai Garden   
1           1          368261001  1.389365  103.886287             NaN   
2           2          369123892  1.385612  103.902577  NTUC Fairprice   
3           3          369127276  1.379434  103.887950           Giant   
4           4          369128055  1.372528  103.893924  NTUC FairPrice   

  Category Tag   Place Type                       Address Information  \
0      tourism   attraction                                       NaN   
1      amenity         cafe                                       NaN   
2         shop  supermarket                                       NaN   
3         shop  supermarket  21, Hougang Street 51, Singapore, 538719   
4         shop  supermarket                                       NaN   

  Contact Details Operational Info Metadata              status  
0             NaN              NaN      NaN       

In [20]:
result_df.head()

,Unnamed: 0,Place ID (OSM ID),lat,lon,Place Name,Category Tag,Place Type,Address Information,Contact Details,Operational Info,Metadata,status
0,0,319830199,1.338296,103.730060,Bonsai Garden,tourism,attraction,NaN,NaN,NaN,NaN,Not Found
1,1,368261001,1.389365,103.886287,NaN,amenity,cafe,NaN,NaN,NaN,NaN,Not Found
2,2,369123892,1.385612,103.902577,NTUC Fairprice,shop,supermarket,NaN,NaN,24/7,NaN,Exists (Geo Match)
3,3,369127276,1.379434,103.887950,Giant,shop,supermarket,"21, Hougang Street 51, Singapore, 538719",NaN,NaN,NaN,Not Found
4,4,369128055,1.372528,103.893924,NTUC FairPrice,shop,supermarket,NaN,NaN,08:00-23:00,NaN,Not Found


In [22]:
print(result_df['status'].unique())

['Not Found' 'Exists (Geo Match)' 'Exists']


In [24]:
# comparing osm and geoapify datasets 
def check_match_v2(row):
    name_val = row['clean_name']
    lat_val1 = row['lat']
    lon_val1 = row['lon']

    # -------------------------
    # Step 1: Name Match First
    # -------------------------
    name_candidates = df2[df2['clean_name'] == name_val]

    if not name_candidates.empty:
        for _, candidate in name_candidates.iterrows():
            lat_val2 = candidate['lat']
            lon_val2 = candidate['lon']

            if pd.isna(lat_val2) or pd.isna(lon_val2) or pd.isna(lat_val1) or pd.isna(lon_val1):
                continue

            dist_val = haversine(lat_val1, lon_val1, lat_val2, lon_val2)

            if dist_val <= THRESHOLD_KM:
                return "Exists (Name + Location Match)"

        return "Name Found (Location Mismatch)"

    # -------------------------
    # Step 2: Geo Match Only
    # -------------------------
    if pd.isna(lat_val1) or pd.isna(lon_val1):
        return "Not Found"

    for _, candidate in df2.iterrows():
        lat_val2 = candidate['lat']
        lon_val2 = candidate['lon']

        if pd.isna(lat_val2) or pd.isna(lon_val2):
            continue

        dist_val = haversine(lat_val1, lon_val1, lat_val2, lon_val2)

        if dist_val <= THRESHOLD_KM:
            return "Exists (Location Match)"

    return "Not Found"

In [25]:
df1['status_v2'] = df1.apply(check_match_v2, axis=1)

In [26]:
result_df_v2 = df1.drop(columns=['clean_name'])
result_df_v2.head()

,Unnamed: 0,Place ID (OSM ID),lat,lon,Place Name,Category Tag,Place Type,Address Information,Contact Details,Operational Info,Metadata,status,status_v2
0,0,319830199,1.338296,103.730060,Bonsai Garden,tourism,attraction,NaN,NaN,NaN,NaN,Not Found,Exists (Name + Location Match)
1,1,368261001,1.389365,103.886287,NaN,amenity,cafe,NaN,NaN,NaN,NaN,Not Found,Not Found
2,2,369123892,1.385612,103.902577,NTUC Fairprice,shop,supermarket,NaN,NaN,24/7,NaN,Exists (Geo Match),Exists (Name + Location Match)
3,3,369127276,1.379434,103.887950,Giant,shop,supermarket,"21, Hougang Street 51, Singapore, 538719",NaN,NaN,NaN,Not Found,Exists (Name + Location Match)
4,4,369128055,1.372528,103.893924,NTUC FairPrice,shop,supermarket,NaN,NaN,08:00-23:00,NaN,Not Found,Exists (Name + Location Match)


In [28]:
print(result_df_v2['status_v2'].unique())

['Exists (Name + Location Match)' 'Not Found' 'Exists (Location Match)'
 'Name Found (Location Mismatch)']


In [29]:
# comparing osm and tomtom api datasets 
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# -----------------------------
# 1. Haversine Distance Function
# -----------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c


# -----------------------------
# 2. Clean Name Function
# -----------------------------
def clean_name(name):
    return str(name).lower().strip()


# Apply cleaning
df1['clean_name'] = df1['Place Name'].apply(clean_name)
df3['clean_name'] = df3['name'].apply(clean_name)


# -----------------------------
# 3. Matching Logic (Updated Safe Version)
# -----------------------------
THRESHOLD_KM = 0.1  # 100 meters


def check_match_v3(row):
    name_val = row['clean_name']
    lat_val1 = row['lat']
    lon_val1 = row['lon']
    place_id_val = row['Place ID (OSM ID)']

    # -------------------------
    # Step 1: Place ID Match
    # -------------------------
    id_candidates = df3[df3['place_id'] == place_id_val]
    if not id_candidates.empty:
        return "Exists (ID Match)"

    # -------------------------
    # Step 2: Name Match + Geo Validation
    # -------------------------
    name_candidates = df3[df3['clean_name'] == name_val]

    if not name_candidates.empty:
        for _, candidate_row in name_candidates.iterrows():
            lat_val2 = candidate_row['lat']
            lon_val2 = candidate_row['lon']

            if pd.isna(lat_val2) or pd.isna(lon_val2) or pd.isna(lat_val1) or pd.isna(lon_val1):
                continue

            distance_val = haversine(lat_val1, lon_val1, lat_val2, lon_val2)

            if distance_val <= THRESHOLD_KM:
                return "Exists (Name + Location Match)"

        return "Name Found (Location Mismatch)"

    # -------------------------
    # Step 3: Geo Match Only
    # -------------------------
    if pd.isna(lat_val1) or pd.isna(lon_val1):
        return "Not Found"

    for _, candidate_row in df3.iterrows():
        lat_val2 = candidate_row['lat']
        lon_val2 = candidate_row['lon']

        if pd.isna(lat_val2) or pd.isna(lon_val2):
            continue

        distance_val = haversine(lat_val1, lon_val1, lat_val2, lon_val2)

        if distance_val <= THRESHOLD_KM:
            return "Exists (Location Match)"

    return "Not Found"


# -----------------------------
# 4. Apply Matching (NEW column)
# -----------------------------
df1['status_v3'] = df1.apply(check_match_v3, axis=1)


# -----------------------------
# 5. Final Output (same format)
# -----------------------------
result_df_v3 = df1.drop(columns=['clean_name'])

print(result_df_v3.head())

   Unnamed: 0  Place ID (OSM ID)       lat         lon      Place Name  \
0           0          319830199  1.338296  103.730060   Bonsai Garden   
1           1          368261001  1.389365  103.886287             NaN   
2           2          369123892  1.385612  103.902577  NTUC Fairprice   
3           3          369127276  1.379434  103.887950           Giant   
4           4          369128055  1.372528  103.893924  NTUC FairPrice   

  Category Tag   Place Type                       Address Information  \
0      tourism   attraction                                       NaN   
1      amenity         cafe                                       NaN   
2         shop  supermarket                                       NaN   
3         shop  supermarket  21, Hougang Street 51, Singapore, 538719   
4         shop  supermarket                                       NaN   

  Contact Details Operational Info Metadata              status  \
0             NaN              NaN      NaN      

In [30]:
result_df_v3.head()

,Unnamed: 0,Place ID (OSM ID),lat,lon,Place Name,Category Tag,Place Type,Address Information,Contact Details,Operational Info,Metadata,status,status_v2,status_v3
0,0,319830199,1.338296,103.730060,Bonsai Garden,tourism,attraction,NaN,NaN,NaN,NaN,Not Found,Exists (Name + Location Match),Not Found
1,1,368261001,1.389365,103.886287,NaN,amenity,cafe,NaN,NaN,NaN,NaN,Not Found,Not Found,Not Found
2,2,369123892,1.385612,103.902577,NTUC Fairprice,shop,supermarket,NaN,NaN,24/7,NaN,Exists (Geo Match),Exists (Name + Location Match),Exists (Location Match)
3,3,369127276,1.379434,103.887950,Giant,shop,supermarket,"21, Hougang Street 51, Singapore, 538719",NaN,NaN,NaN,Not Found,Exists (Name + Location Match),Not Found
4,4,369128055,1.372528,103.893924,NTUC FairPrice,shop,supermarket,NaN,NaN,08:00-23:00,NaN,Not Found,Exists (Name + Location Match),Not Found


In [31]:
print(result_df_v3['status_v3'].unique())

['Not Found' 'Exists (Location Match)' 'Exists (Name + Location Match)'
 'Name Found (Location Mismatch)']


In [39]:
# checking confidence and final status 
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# -----------------------------
# 1. Distance Function (renamed)
# -----------------------------
def geo_distance_calc(lat_a, lon_a, lat_b, lon_b):
    R = 6371
    lat_a, lon_a, lat_b, lon_b = map(radians, [lat_a, lon_a, lat_b, lon_b])
    
    dlat = lat_b - lat_a
    dlon = lon_b - lon_a
    
    a = sin(dlat/2)**2 + cos(lat_a) * cos(lat_b) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c


# -----------------------------
# 2. Confidence Function (renamed)
# -----------------------------
def compute_confidence_v4(row_data):
    
    status_text = str(row_data.get("status_v3", ""))

    # Name Score
    if "Name + Location Match" in status_text:
        name_score_val = 1
    elif "Name Found" in status_text:
        name_score_val = 0.7
    else:
        name_score_val = 0

    # Location Score
    if "Location Match" in status_text:
        location_score_val = 1
    elif "Mismatch" in status_text:
        location_score_val = 0.3
    else:
        location_score_val = 0

    # Metadata Score
    metadata_score_val = 1 if pd.notna(row_data.get("contact details")) else 0.3

    # Category Score
    category_score_val = 1 if pd.notna(row_data.get("Place Type")) else 0.5

    # Final Confidence
    final_conf = (
        0.4 * name_score_val +
        0.4 * location_score_val +
        0.1 * metadata_score_val +
        0.1 * category_score_val
    )

    return round(final_conf, 2)


# -----------------------------
# 3. Final Status Function (renamed)
# -----------------------------
def derive_final_status_v4(row_data):
    
    status_text = str(row_data.get("status_v3", ""))

    if "Name + Location Match" in status_text:
        return "EXISTS"
    
    elif "Location Match" in status_text:
        return "EXISTS"
    
    elif "Name Found (Location Mismatch)" in status_text:
        return "CLOSED"
    
    elif "Not Found" in status_text:
        return "CANNOT_SAY"
    
    else:
        return "REVIEW"


# -----------------------------
# 4. Main Wrapper Function (renamed)
# -----------------------------
def build_final_output_v4(input_df):
    
    temp_df = input_df.copy()   # keeps original safe
    
    # New column names (no overwrite)
    temp_df["confidence_v4"] = temp_df.apply(compute_confidence_v4, axis=1)
    temp_df["final_status_v4"] = temp_df.apply(derive_final_status_v4, axis=1)
    
    return temp_df


# -----------------------------
# 5. Execute (safe output df)
# -----------------------------
result_df_v4 = build_final_output_v4(result_df_v3)

# Preview
print(result_df_v4.head())

   Unnamed: 0  Place ID (OSM ID)       lat         lon      Place Name  \
0           0          319830199  1.338296  103.730060   Bonsai Garden   
1           1          368261001  1.389365  103.886287             NaN   
2           2          369123892  1.385612  103.902577  NTUC Fairprice   
3           3          369127276  1.379434  103.887950           Giant   
4           4          369128055  1.372528  103.893924  NTUC FairPrice   

  Category Tag   Place Type                       Address Information  \
0      tourism   attraction                                       NaN   
1      amenity         cafe                                       NaN   
2         shop  supermarket                                       NaN   
3         shop  supermarket  21, Hougang Street 51, Singapore, 538719   
4         shop  supermarket                                       NaN   

  Contact Details Operational Info Metadata              status  \
0             NaN              NaN      NaN      

In [40]:
result_df_v4.head()

,Unnamed: 0,Place ID (OSM ID),lat,lon,Place Name,Category Tag,Place Type,Address Information,Contact Details,Operational Info,Metadata,status,status_v2,status_v3,confidence_v4,final_status_v4
0,0,319830199,1.338296,103.730060,Bonsai Garden,tourism,attraction,NaN,NaN,NaN,NaN,Not Found,Exists (Name + Location Match),Not Found,0.13,CANNOT_SAY
1,1,368261001,1.389365,103.886287,NaN,amenity,cafe,NaN,NaN,NaN,NaN,Not Found,Not Found,Not Found,0.13,CANNOT_SAY
2,2,369123892,1.385612,103.902577,NTUC Fairprice,shop,supermarket,NaN,NaN,24/7,NaN,Exists (Geo Match),Exists (Name + Location Match),Exists (Location Match),0.53,EXISTS
3,3,369127276,1.379434,103.887950,Giant,shop,supermarket,"21, Hougang Street 51, Singapore, 538719",NaN,NaN,NaN,Not Found,Exists (Name + Location Match),Not Found,0.13,CANNOT_SAY
4,4,369128055,1.372528,103.893924,NTUC FairPrice,shop,supermarket,NaN,NaN,08:00-23:00,NaN,Not Found,Exists (Name + Location Match),Not Found,0.13,CANNOT_SAY


In [41]:
result_df_v4['final_status_v4'].unique()

array(['CANNOT_SAY', 'EXISTS', 'CLOSED'], dtype=object)

In [42]:
result_df_v4['confidence_v4'].unique()

array([0.13, 0.53, 0.93])

In [43]:
result_df_v4["final_status_v4"].value_counts()

final_status_v4
CANNOT_SAY    7679
EXISTS         515
CLOSED         416
Name: count, dtype: int64

In [47]:
closed_rows_v4 = result_df_v4[result_df_v4["final_status_v4"] == "CLOSED"]

In [49]:
closed_rows_v4

,Unnamed: 0,Place ID (OSM ID),lat,lon,Place Name,Category Tag,Place Type,Address Information,Contact Details,Operational Info,Metadata,status,status_v2,status_v3,confidence_v4,final_status_v4
113,113,1316097225,1.251236,103.817730,The Coffee Bean & Tea Leaf,amenity,cafe,NaN,NaN,NaN,NaN,Exists,Exists (Name + Location Match),Name Found (Location Mismatch),0.53,CLOSED
144,144,1699579420,1.373829,103.845554,Jack's Place,amenity,restaurant,NaN,NaN,NaN,NaN,Exists,Exists (Name + Location Match),Name Found (Location Mismatch),0.53,CLOSED
156,156,1703872735,1.387554,103.869249,The Coffee Bean & Tea Leaf,amenity,cafe,"1, Seletar Rd, Greenwich V, Singapore, 807011",NaN,Mo-Su 08:00-23:00,NaN,Exists,Exists (Name + Location Match),Name Found (Location Mismatch),0.53,CLOSED
165,165,1711105942,1.330649,103.795681,The Coffee Bean & Tea Leaf,amenity,cafe,"#01-04, 1 Fifth Avenue, Guthrie House, Singapo...",https://www.coffeebean.com.sg/,NaN,NaN,Exists,Exists (Name + Location Match),Name Found (Location Mismatch),0.53,CLOSED
200,200,1844016090,1.307780,103.818364,Fusion Spoon,amenity,restaurant,"1, Cluny Road, 259569",https://fusion-spoon-at-botanic-gardens.busine...,NaN,NaN,Exists,Exists (Name + Location Match),Name Found (Location Mismatch),0.53,CLOSED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8398,8398,764586886,NaN,NaN,Quayside Isle,shop,mall,"31, Ocean Way, Singapore, 098375",https://www.quaysideisle.com/,NaN,NaN,Exists,Name Found (Location Mismatch),Name Found (Location Mismatch),0.53,CLOSED
8529,8529,1346557962,NaN,NaN,Tenderbest Makcik Tuckshop,amenity,restaurant,"97, Hougang Avenue 8, Singapore, 538768",NaN,NaN,NaN,Exists,Name Found (Location Mismatch),Name Found (Location Mismatch),0.53,CLOSED
8593,8593,8308880,NaN,NaN,Crave,amenity,restaurant,NaN,NaN,NaN,NaN,Exists,Name Found (Location Mismatch),Name Found (Location Mismatch),0.53,CLOSED
8607,8607,19216100,NaN,NaN,The Coffee Bean & Tea Leaf,amenity,cafe,NaN,NaN,NaN,NaN,Exists,Name Found (Location Mismatch),Name Found (Location Mismatch),0.53,CLOSED


In [50]:
closed_rows_v4.to_csv("closed_rows_v4.csv", index=False)